Imports needed to load the model and generate text.

In [1]:
import random, pickle

import numpy as np

import tensorflow as tf
from tensorflow.keras.models import load_model

I0000 00:00:1795149570.709516   93340 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795149570.709932   93340 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795149570.753513   93340 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795149571.627358   93340 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Load the trained model, vocabulary, and tokenizer.

In [2]:
from nltk.tokenize import RegexpTokenizer

model = load_model('../models/keras_next_word_model.keras')

with open('../data/processed/unique_tokens.p', 'rb') as f:
    unique_tokens = pickle.load(f)

with open('../data/processed/unique_token_idx.p', 'rb') as f:
    unique_token_idx = pickle.load(f)

tokenizer = RegexpTokenizer(r'\w+')

E0000 00:00:1795149593.200768   93340 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1795149593.201074   93956 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1795149593.217135   93340 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Predict the most likely next words for a short phrase.

In [3]:
def predict_next_word(input_text, n_best=1):
    words = input_text.lower().split()[-10:]

    token_ids = [unique_token_idx[word] for word in words]
    token_ids = [0] * (10 - len(token_ids)) + token_ids

    X = np.array([token_ids])

    predictions = model.predict(X, verbose=0)[0]

    best_indices = np.argsort(predictions)[-n_best:][::-1]

    return best_indices

Show the top five predictions after "I will".

In [4]:
possible = predict_next_word("I will", 5)

for idx in possible:
    print(unique_tokens[idx])

be
have
help
only
take


Repeat prediction to build a longer text continuation.

In [5]:
def generate_text(input_text, n_words, creativity=3):
    word_sequence = input_text.split()
    current = 0
    for _ in range(n_words):
        sub_sequence = " ".join(tokenizer.tokenize(" ".join(word_sequence).lower())[current:current+n_words])
        try:
            choice = unique_tokens[random.choice(predict_next_word(sub_sequence, creativity))]
        except:
            choice = random.choice(unique_tokens)
        word_sequence.append(choice)
        current += 1
    return " ".join(word_sequence)

Generate fifteen words starting from "I will".

In [6]:
generate_text("I will", 15, 2)

'I will be a military text q positions on wednesday january claimed are reportedly killed on facebook'